# Retail Demand Forecasting — XGBoost → RDS

**Pipeline**
1. Load `retail_store_inventory.csv` from S3
2. Feature engineering (clean — no data leakage)
3. Chronological train / test split (split: 2023-11-01)
4. XGBoost tuned via RandomizedSearchCV + TimeSeriesSplit
5. Write actuals + predictions + metrics to RDS MySQL
6. Sanity-check read-back

**Run environment:** SageMaker Studio / SageMaker Notebook (Python 3, `ml.m5.large` or larger)


## 0. Install dependencies
SageMaker base images include scikit-learn and boto3. We only need to add xgboost and pymysql.

## 1. Configuration — fill in your values

In [ ]:
!pip install pandas numpy prophet boto3 sqlalchemy pymysql

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.1/12.1 MB 137.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 115.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 160.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 615.1/615.1 kB 51.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8/8 [prophet]m7/8 [prophet]y]]esources]


In [ ]:
import warnings, time, io, logging
from datetime import datetime, timezone

warnings.filterwarnings('ignore')

import boto3
import numpy as np
import pandas as pd
# from xgboost import XGBRegressor


from sklearn.model_selection import RandomizedSearchCV, TimeSeriesSplit
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sqlalchemy import create_engine, text

# ── S3 ──────────────────────────────────────────────────────────────────
S3_BUCKET   = 'mlba-storedata-carinaz'
S3_DATA_KEY = 'retail_store_inventory.csv'

# ── RDS MySQL ────────────────────────────────────────────────────────────
RDS_HOST     = 'churn-db.cbwqawzgmz8b.us-east-1.rds.amazonaws.com'
RDS_PORT     = '3306'
RDS_DB       = 'retail_forecast'
RDS_USER     = 'admin'
RDS_PASSWORD = 'awscarina.'


# ── Model ────────────────────────────────────────────────────────────────
SPLIT_DATE   = pd.Timestamp('2023-11-01')
RANDOM_STATE = 42
N_ITER       = 20          # random search trials; increase for better tuning

RUN_TS = datetime.now(timezone.utc).replace(microsecond=0, tzinfo=None)
print(f'Run timestamp : {RUN_TS}')
print(f'Split date    : {SPLIT_DATE.date()}')

Run timestamp : 2026-04-19 04:59:21
Split date    : 2023-11-01


In [ ]:
!pip install xgboost==1.7.6

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 200.3/200.3 MB 57.7 MB/s  0:00:03m0:00:0100:01


In [ ]:
from xgboost import XGBRegressor

## 2. Load data from S3

In [ ]:
s3  = boto3.client('s3')
obj = s3.get_object(Bucket=S3_BUCKET, Key=S3_DATA_KEY)
df  = pd.read_csv(io.BytesIO(obj['Body'].read()), parse_dates=['Date'])
df  = df.sort_values(['Store ID', 'Product ID', 'Date']).reset_index(drop=True)

print(f'Rows  : {df.shape[0]:,}')
print(f'Cols  : {df.shape[1]}')
print(f'Dates : {df["Date"].min().date()} → {df["Date"].max().date()}')
print(f'Stores: {sorted(df["Store ID"].unique())}')
print(f'SKUs  : {df["Product ID"].nunique()}')
df.head(3)

Rows  : 73,100
Cols  : 15
Dates : 2022-01-01 → 2024-01-01
Stores: ['S001', 'S002', 'S003', 'S004', 'S005']
SKUs  : 20


,Date,Store ID,Product ID,Category,Region,Inventory Level,Units Sold,Units Ordered,Demand Forecast,Price,Discount,Weather Condition,Holiday/Promotion,Competitor Pricing,Seasonality
0,2022-01-01,S001,P0001,Groceries,North,231,127,55,135.47,33.50,20,Rainy,0,29.69,Autumn
1,2022-01-02,S001,P0001,Groceries,West,116,81,104,92.94,27.95,10,Cloudy,0,30.89,Spring
2,2022-01-03,S001,P0001,Electronics,West,154,5,189,5.36,62.70,20,Rainy,0,58.22,Winter


## 3. Feature engineering

Clean feature set — `Demand Forecast`, `Inventory Level`, `Units Ordered` excluded to prevent leakage.

In [ ]:
def prepare_features(data: pd.DataFrame) -> pd.DataFrame:
    temp = data.copy()

    # Calendar
    temp['Year']       = temp['Date'].dt.year
    temp['Month']      = temp['Date'].dt.month
    temp['Day']        = temp['Date'].dt.day
    temp['DayOfWeek']  = temp['Date'].dt.dayofweek
    temp['Is_Weekend'] = temp['DayOfWeek'].isin([5, 6]).astype(int)

    # Lag / rolling (grouped by Store × Product — no cross-series leakage)
    grp = temp.groupby(['Store ID', 'Product ID'])['Units Sold']
    temp['Lag_1']          = grp.shift(1)
    temp['Lag_7']          = grp.shift(7)
    temp['Rolling_Mean_7'] = grp.transform(lambda x: x.shift(1).rolling(7).mean())
    temp['Rolling_Std_7']  = grp.transform(lambda x: x.shift(1).rolling(7).std())

    # Categorical encodings
    temp['Category_Code'] = temp['Category'].astype('category').cat.codes
    temp['Weather_Code']  = temp['Weather Condition'].astype('category').cat.codes

    return temp.dropna().reset_index(drop=True)


df_ml = prepare_features(df)

FEATURES = [
    'Price', 'Discount', 'Holiday/Promotion', 'Competitor Pricing',
    'Year', 'Month', 'Day', 'DayOfWeek', 'Is_Weekend',
    'Lag_1', 'Lag_7', 'Rolling_Mean_7', 'Rolling_Std_7',
    'Category_Code', 'Weather_Code',
]
TARGET = 'Units Sold'

print(f'Modeling rows: {df_ml.shape[0]:,}  |  Features: {len(FEATURES)}')

Modeling rows: 72,400  |  Features: 15


## 4. Chronological train / test split

In [ ]:
train = df_ml[df_ml['Date'] < SPLIT_DATE].copy()
test  = df_ml[df_ml['Date'] >= SPLIT_DATE].copy()

X_train, y_train = train[FEATURES], train[TARGET]
X_test,  y_test  = test[FEATURES],  test[TARGET]

print(f'Train: {len(train):,} rows  ({train["Date"].min().date()} → {train["Date"].max().date()})')
print(f'Test : {len(test):,} rows  ({test["Date"].min().date()} → {test["Date"].max().date()})')

Train: 66,200 rows  (2022-01-08 → 2023-10-31)
Test : 6,200 rows  (2023-11-01 → 2024-01-01)


## 5. XGBoost — RandomizedSearchCV + TimeSeriesSplit

In [ ]:
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error
import numpy as np

best_xgb = XGBRegressor(
    n_estimators=300,
    subsample= 0.9,
    min_child_weight=1,
    max_depth=6,
    learning_rate=0.01,
    random_state=42,
    colsample_bytree=0.7,
    n_jobs=-1
)

# ===== 2. 训练 =====
best_xgb.fit(X_train, y_train)

# ===== 3. 预测 =====
y_pred = best_xgb.predict(X_test)

# ===== 4. 简单评估 =====
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
print(f"RMSE: {rmse:.4f}")

RMSE: 108.1199


## 6. Generate predictions + residual-based 80% CI

In [ ]:
# Test-set predictions
test = test.copy()
test['predicted_units'] = best_xgb.predict(X_test).clip(min=0)

# Residual-based 80% CI (per-record)
residuals = test[TARGET].values - test['predicted_units'].values
ci_lo = float(np.percentile(residuals, 10))
ci_hi = float(np.percentile(residuals, 90))
test['predicted_lower'] = (test['predicted_units'] + ci_lo).clip(lower=0)
test['predicted_upper'] = (test['predicted_units'] + ci_hi).clip(lower=0)

# Metrics
rmse = float(np.sqrt(mean_squared_error(test[TARGET], test['predicted_units'])))
mae  = float(mean_absolute_error(test[TARGET], test['predicted_units']))
r2   = float(r2_score(test[TARGET], test['predicted_units']))
nz   = test[TARGET] > 0
mape = float(np.mean(np.abs((test.loc[nz, TARGET] - test.loc[nz, 'predicted_units']) / test.loc[nz, TARGET])) * 100)

print(f'RMSE : {rmse:.2f}')
print(f'MAE  : {mae:.2f}')
print(f'R²   : {r2:.4f}')
print(f'MAPE : {mape:.2f}%')
print(f'CI   : [{ci_lo:.1f}, {ci_hi:.1f}] per record (80%)')

RMSE : 108.12
MAE  : 87.73
R²   : -0.0055
MAPE : 314.83%
CI   : [-119.7, 220.5] per record (90%)


In [ ]:
test

,Date,Store ID,Product ID,Category,Region,Inventory Level,Units Sold,Units Ordered,Demand Forecast,Price,...,Is_Weekend,Lag_1,Lag_7,Rolling_Mean_7,Rolling_Std_7,Category_Code,Weather_Code,predicted_units,predicted_lower,predicted_upper
662,2023-11-01,S001,P0001,Groceries,North,110,15,81,31.36,62.27,...,0,311.0,27.0,195.714286,132.062756,3,1,129.350677,9.655403,349.837891
663,2023-11-02,S001,P0001,Furniture,West,386,139,123,147.80,70.51,...,0,15.0,177.0,194.000000,134.669967,2,3,130.901794,11.206520,351.389008
664,2023-11-03,S001,P0001,Groceries,North,242,136,186,148.20,75.65,...,0,139.0,20.0,188.571429,136.226352,3,0,130.624573,10.929298,351.111786
665,2023-11-04,S001,P0001,Clothing,South,287,180,62,193.64,35.50,...,1,136.0,255.0,205.142857,118.160101,0,0,133.171082,13.475807,353.658295
666,2023-11-05,S001,P0001,Furniture,South,166,21,186,37.23,54.31,...,1,180.0,218.0,194.428571,116.271030,2,2,132.238052,12.542778,352.725281
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
72395,2023-12-28,S005,P0020,Groceries,South,198,56,27,50.18,21.75,...,0,138.0,160.0,133.714286,85.904432,3,3,128.821884,9.126610,349.309082
72396,2023-12-29,S005,P0020,Clothing,East,446,268,30,267.54,85.58,...,0,56.0,228.0,118.857143,89.518022,0,3,129.744431,10.049156,350.231628
72397,2023-12-30,S005,P0020,Toys,North,251,149,181,162.92,79.48,...,1,268.0,220.0,124.571429,98.474797,4,0,127.985596,8.290321,348.472809
72398,2023-12-31,S005,P0020,Furniture,East,64,40,99,59.69,90.79,...,1,149.0,6.0,114.428571,90.326919,2,2,120.509911,0.814636,340.997131


## 7. Connect to RDS MySQL and create schema

In [ ]:
# Create DB if it doesn't exist
server_engine = create_engine(
    f'mysql+pymysql://{RDS_USER}:{RDS_PASSWORD}@{RDS_HOST}:{RDS_PORT}',
    pool_pre_ping=True
)
with server_engine.begin() as conn:
    conn.execute(text(
        f'CREATE DATABASE IF NOT EXISTS `{RDS_DB}` '
        'CHARACTER SET utf8mb4 COLLATE utf8mb4_unicode_ci'
    ))
server_engine.dispose()
print(f'Database `{RDS_DB}` ready')

# Working engine
engine = create_engine(
    f'mysql+pymysql://{RDS_USER}:{RDS_PASSWORD}@{RDS_HOST}:{RDS_PORT}/{RDS_DB}',
    pool_pre_ping=True
)
with engine.begin() as conn:
    conn.execute(text('SELECT 1'))
print('RDS connection OK')

Database `retail_forecast` ready
RDS connection OK


## 8. Create / reset tables

Three tables:
- **`actuals`** — ground truth daily sales per Store × Product
- **`predictions`** — XGBoost forecasts with 80% CI
- **`metrics`** — per-run model performance summary

In [ ]:
DDL = [
    '''
    CREATE TABLE IF NOT EXISTS actuals (
        store_id        VARCHAR(16)  NOT NULL,
        product_id      VARCHAR(16)  NOT NULL,
        category        VARCHAR(64)  NOT NULL,
        region          VARCHAR(32)  NOT NULL,
        obs_date        DATE         NOT NULL,
        units_sold      DOUBLE       NOT NULL,
        price           DOUBLE,
        discount        DOUBLE,
        is_promo        TINYINT,
        run_timestamp   DATETIME     NOT NULL,
        PRIMARY KEY (store_id, product_id, obs_date)
    )
    ''',
    '''
    CREATE TABLE IF NOT EXISTS predictions (
        store_id            VARCHAR(16)  NOT NULL,
        product_id          VARCHAR(16)  NOT NULL,
        category            VARCHAR(64)  NOT NULL,
        region              VARCHAR(32)  NOT NULL,
        obs_date            DATE         NOT NULL,
        actual_units        DOUBLE       NOT NULL,
        predicted_units     DOUBLE       NOT NULL,
        predicted_lower     DOUBLE       NOT NULL,
        predicted_upper     DOUBLE       NOT NULL,
        abs_error           DOUBLE       NOT NULL,
        run_timestamp       DATETIME     NOT NULL,
        PRIMARY KEY (store_id, product_id, obs_date, run_timestamp)
    )
    ''',
    '''
    CREATE TABLE IF NOT EXISTS metrics (
        model_name      VARCHAR(64)  NOT NULL,
        rmse            DOUBLE       NOT NULL,
        mae             DOUBLE       NOT NULL,
        r2              DOUBLE       NOT NULL,
        mape_pct        DOUBLE       NOT NULL,
        ci_lower_bound  DOUBLE       NOT NULL,
        ci_upper_bound  DOUBLE       NOT NULL,
        train_time_s    DOUBLE       NOT NULL,
        n_test_rows     INT          NOT NULL,
        split_date      DATE         NOT NULL,
        best_params     TEXT,
        run_timestamp   DATETIME     NOT NULL,
        PRIMARY KEY (model_name, run_timestamp)
    )
    '''
]

with engine.begin() as conn:
    for stmt in DDL:
        conn.execute(text(stmt))

print('Schema ready (tables created if they did not exist)')

Schema ready (tables created if they did not exist)


## 9. Write actuals to RDS

In [ ]:
actuals_df = test[[
    'Store ID', 'Product ID', 'Category', 'Region',
    'Date', TARGET, 'Price', 'Discount', 'Holiday/Promotion'
]].copy()

actuals_df.columns = [
    'store_id', 'product_id', 'category', 'region',
    'obs_date', 'units_sold', 'price', 'discount', 'is_promo'
]
actuals_df['run_timestamp'] = RUN_TS

# Upsert: delete existing rows for these store/product/date combos, then insert
with engine.begin() as conn:
    conn.execute(text('TRUNCATE TABLE actuals'))

actuals_df.to_sql('actuals', engine, if_exists='append', index=False, chunksize=2000)
print(f'Wrote {len(actuals_df):,} actuals rows')

Wrote 6,200 actuals rows


## 10. Write predictions to RDS

In [ ]:
preds_df = test[[
    'Store ID', 'Product ID', 'Category', 'Region', 'Date',
    TARGET, 'predicted_units', 'predicted_lower', 'predicted_upper'
]].copy()

preds_df.columns = [
    'store_id', 'product_id', 'category', 'region', 'obs_date',
    'actual_units', 'predicted_units', 'predicted_lower', 'predicted_upper'
]
preds_df['abs_error']     = (preds_df['actual_units'] - preds_df['predicted_units']).abs()
preds_df['run_timestamp'] = RUN_TS

with engine.begin() as conn:
    conn.execute(text('DELETE FROM predictions WHERE run_timestamp = :ts'), {'ts': RUN_TS})

preds_df.to_sql('predictions', engine, if_exists='append', index=False, chunksize=2000)
print(f'Wrote {len(preds_df):,} prediction rows')

Wrote 6,200 prediction rows


## 12. Sanity check — read back from RDS

In [ ]:
# Summary by store
summary = pd.read_sql(text('''
    SELECT
        store_id,
        COUNT(*)                              AS n_rows,
        ROUND(AVG(actual_units), 2)           AS avg_actual,
        ROUND(AVG(predicted_units), 2)        AS avg_predicted,
        ROUND(SQRT(AVG(POW(abs_error,2))),2)  AS rmse
    FROM predictions
    WHERE run_timestamp = :ts
    GROUP BY store_id
    ORDER BY store_id
'''), engine, params={'ts': RUN_TS})

print('Predictions per store (latest run):')
display(summary)

Predictions per store (latest run):


,store_id,n_rows,avg_actual,avg_predicted,rmse
0,S001,1240,134.96,129.19,105.18
1,S002,1240,135.02,129.23,108.97
2,S003,1240,140.13,129.33,108.89
3,S004,1240,134.18,129.38,106.84
4,S005,1240,139.62,129.05,110.63


In [ ]:
# Spot-check: one store + product
check = pd.read_sql(text('''
    SELECT
        p.obs_date,
        p.actual_units,
        p.predicted_units,
        p.predicted_lower,
        p.predicted_upper,
        p.abs_error
    FROM predictions p
    WHERE p.store_id       = :store
      AND p.product_id     = :product
      AND p.run_timestamp  = :ts
    ORDER BY p.obs_date
    LIMIT 10
'''), engine, params={'store': 'S001', 'product': 'P0001', 'ts': RUN_TS})

print('Spot-check S001 / P0001:')
display(check)

Spot-check S001 / P0001:


,obs_date,actual_units,predicted_units,predicted_lower,predicted_upper,abs_error
0,2023-11-01,15.0,129.350677,9.655403,349.837891,114.350677
1,2023-11-02,139.0,130.901794,11.206520,351.389008,8.098206
2,2023-11-03,136.0,130.624573,10.929298,351.111786,5.375427
3,2023-11-04,180.0,133.171082,13.475807,353.658295,46.828918
4,2023-11-05,21.0,132.238052,12.542778,352.725281,111.238052
5,2023-11-06,49.0,128.357437,8.662163,348.844666,79.357437
6,2023-11-07,22.0,131.161118,11.465843,351.648315,109.161118
7,2023-11-08,324.0,130.382675,10.687401,350.869873,193.617325
8,2023-11-09,75.0,132.316086,12.620811,352.803284,57.316086
9,2023-11-10,234.0,129.783615,10.088341,350.270813,104.216385


## 13. Useful SQL for Streamlit app

These queries are what `streamlit_app.py` uses — kept here as reference.

In [ ]:
STREAMLIT_QUERIES = {

# Forecast explorer: daily totals with CI
'forecast_daily': '''
    SELECT
        p.obs_date,
        p.store_id,
        p.product_id,
        p.category,
        p.region,
        p.actual_units,
        p.predicted_units,
        p.predicted_lower,
        p.predicted_upper,
        p.abs_error,
        a.price,
        a.discount,
        a.is_promo
    FROM predictions p
    JOIN actuals a
      ON  a.store_id   = p.store_id
      AND a.product_id = p.product_id
      AND a.obs_date   = p.obs_date
    WHERE p.run_timestamp = (SELECT MAX(run_timestamp) FROM predictions)
    ORDER BY p.obs_date, p.store_id, p.product_id
''',

# Inventory alerts: stock vs 7-day rolling forecast demand
'inventory_alerts': '''
    SELECT
        a.store_id,
        a.product_id,
        a.category,
        a.region,
        a.obs_date,
        a.units_sold      AS current_stock_proxy,
        p.predicted_units AS forecast_demand,
        ROUND(a.units_sold / NULLIF(p.predicted_units, 0), 1) AS days_of_supply
    FROM actuals a
    JOIN predictions p
      ON  p.store_id   = a.store_id
      AND p.product_id = a.product_id
      AND p.obs_date   = a.obs_date
    WHERE p.run_timestamp = (SELECT MAX(run_timestamp) FROM predictions)
      AND a.obs_date      = (SELECT MAX(obs_date) FROM actuals)
    ORDER BY days_of_supply ASC
'''

}

for name, sql in STREAMLIT_QUERIES.items():
    print(f'--- {name} ---')
    df_check = pd.read_sql(text(sql), engine)
    print(f'  {len(df_check):,} rows')
    display(df_check.head(3))
    print()

--- forecast_daily ---
  6,200 rows


,obs_date,store_id,product_id,category,region,actual_units,predicted_units,predicted_lower,predicted_upper,abs_error,price,discount,is_promo
0,2023-11-01,S001,P0001,Groceries,North,15.0,129.350677,9.655403,349.837891,114.350677,62.27,5.0,1
1,2023-11-01,S001,P0002,Furniture,East,58.0,130.036591,10.341316,350.523804,72.036591,45.93,15.0,1
2,2023-11-01,S001,P0003,Furniture,East,58.0,129.475479,9.780205,349.962708,71.475479,49.92,15.0,1



--- inventory_alerts ---
  100 rows


,store_id,product_id,category,region,obs_date,current_stock_proxy,forecast_demand,days_of_supply
0,S005,P0020,Groceries,East,2024-01-01,6.0,127.894516,0.0
1,S004,P0004,Furniture,South,2024-01-01,6.0,132.132645,0.0
2,S004,P0013,Electronics,West,2024-01-01,6.0,128.975830,0.0
